# ArtifactGemmaTest
Independent sequence test: Magi text detection + Magi/Paddle OCR → local Gemma Text/Artifact classification → visual review.

Edit the sequence and page settings below, then Run All in `godfather.venv`. Existing raw OCR caches are reused by image checksum; missing OCR is generated locally. **No agreement-based deletion**: every detected text box reaches this test. Punctuation-only expressions are retained directly; empty Paddle OCR uses Magi punctuation only, otherwise the box is deleted; unrecognized Gemma answers become Review. Only Paddle OCR is given to Gemma; Magi OCR is retained for visual comparison only. The classifier sees text only, not the crop. Results are experimental; inspect them visually. No production pipeline or existing cache is modified.

In [1]:
import os, sys, json, hashlib, gc, math, re, unicodedata
os.environ['HF_HUB_OFFLINE']='1'
os.environ['TRANSFORMERS_OFFLINE']='1'
os.environ['PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK']='True'
import torch  # Import before Paddle on this machine.
from pathlib import Path
from difflib import SequenceMatcher
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image
from IPython.display import display

PROJECT=Path('C:/Users/ESHAAN/BYTE/Applied-Task/Magi')
SPLIT='development'
SEQUENCE_ID='seq_5e4a9d63617d62bf'
PAGE_NUMBER=None  # None = all pages; 1, 2, or 3 = one page.
MAGI_MODEL=Path('C:/MODELS/magiv2')
GEMMA_MODEL=Path('C:/MODELS/gemma-3-4b-it')
PADDLE_DET_PATH=Path.home()/'.paddlex/official_models/PP-OCRv5_server_det'
PADDLE_REC_PATH=Path.home()/'.paddlex/official_models/en_PP-OCRv5_mobile_rec'
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
CACHE=PROJECT/'paddleOCR-Magi/cache_artifact_gemma_test'
FORCE_GEMMA=False  # True reruns classification, keeping OCR caches.
EXPERIMENT=PROJECT/'FINETUNING-EXPERMENTS/NarrationModelwBubbles'
CACHE_ROOTS=[PROJECT/'paddleOCR-Magi/cache_contour_full_inference',
             EXPERIMENT/'cache_selected_ocr/full_pipeline',
             PROJECT/'paddleOCR-Magi/cache_inference']
print('Kernel:',sys.executable)


Kernel: c:\Users\ESHAAN\BYTE\godfather.venv\Scripts\python.exe


## 1. Locate pages and discover reusable OCR caches
Spaces pasted into settings are stripped. If the sequence belongs to another split, its unique matching folder is selected automatically.

In [2]:
# Ignore whitespace accidentally pasted with sequence IDs or split names.
SEQUENCE_ID=SEQUENCE_ID.strip()
SPLIT=SPLIT.strip()
sequence_folder=PROJECT/'TaskData'/SPLIT/'images'/SEQUENCE_ID
if not sequence_folder.is_dir():
    matches=[p for p in (PROJECT/'TaskData').glob('*/images/'+SEQUENCE_ID) if p.is_dir()]
    if len(matches)==1:
        sequence_folder=matches[0]
        SPLIT=sequence_folder.parent.parent.name
        print('Found sequence in split:',SPLIT)
    else:
        raise FileNotFoundError(f'Sequence folder not found: {sequence_folder}. Check SEQUENCE_ID and PROJECT. Matches: {matches}')
PAGE_FILES=sorted(p for p in sequence_folder.iterdir() if p.suffix.lower()=='.png')
available_pages=[p.name for p in PAGE_FILES]
if PAGE_NUMBER is not None:
    PAGE_NUMBER=int(PAGE_NUMBER)
    PAGE_FILES=[p for p in PAGE_FILES if p.stem.isdigit() and int(p.stem)==PAGE_NUMBER]
if not PAGE_FILES:
    raise FileNotFoundError(f'No selected PNG pages in {sequence_folder}. PAGE_NUMBER={PAGE_NUMBER}; available pages={available_pages}')
CACHE.mkdir(parents=True,exist_ok=True)
def file_hash(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def digest(value):return hashlib.sha256(json.dumps(value,sort_keys=True,ensure_ascii=True,allow_nan=False).encode()).hexdigest()
def read_payload(path):
    payload=json.loads(Path(path).read_text(encoding='utf-8'))
    assert digest(payload['data'])==payload['data_hash'],f'Invalid cache checksum: {path}'
    return payload['data']
print('Kernel:',sys.executable)
print('Pages:',[p.name for p in PAGE_FILES])


Kernel: c:\Users\ESHAAN\BYTE\godfather.venv\Scripts\python.exe
Pages: ['01.png', '02.png', '03.png']


In [3]:
def cached_page(path):
    sha=file_hash(path)
    with Image.open(path) as im: size=im.size
    for root in CACHE_ROOTS:
        for manifest_path in sorted(root.rglob('manifest.json')):
            try:
                manifest=json.loads(manifest_path.read_text(encoding='utf-8'))
                matches=[i for i,p in enumerate(manifest['inputs']['pages'])
                         if p['sha256']==sha and (p['width'],p['height'])==size]
                if len(matches)!=1:continue
                index=matches[0]
                selection_path=manifest_path.with_name('ocr_selection.json')
                if selection_path.is_file():
                    page=read_payload(selection_path)[index]
                    if all(v['text_index']==i for i,v in enumerate(page)):
                        return [dict(box=v['box'],magi_text=v['magi_text'],paddle_text=v['paddle_text'],
                            paddle_lines=v.get('paddle_lines',[]),selected_text=v.get('selected_text',''),
                            paddle_status=v.get('paddle_status','cached'),paddle_error=v.get('paddle_error')) for v in page],str(selection_path)
                raw=read_payload(manifest_path.with_name('magi_raw.json'))[index]
                paddle=read_payload(manifest_path.with_name('paddle_ocr.json'))[index]
                if len(raw['texts'])!=len(paddle):continue
                if not all(v['box']==b for v,b in zip(paddle,raw['texts'])):continue
                return [dict(box=b,magi_text=t,paddle_text=v['text'],paddle_lines=v['lines'],
                    selected_text=None,paddle_status=v['status'],paddle_error=v.get('error'))
                    for b,t,v in zip(raw['texts'],raw['ocr'],paddle)],str(manifest_path.parent)
            except (OSError,KeyError,ValueError,AssertionError,IndexError,TypeError):
                continue
    return None,None

pages={}
for path in PAGE_FILES:
    records,source=cached_page(path)
    if records is not None:pages[str(path)]=dict(records=records,source=source)
    print('CACHE' if records is not None else 'MISSING',path.name,source or '')


MISSING 01.png 
MISSING 02.png 
MISSING 03.png 


## 2. Create missing OCR only
Reuse the existing Magi/Paddle extraction code. Magi finds the boxes; Paddle reads each box crop. Models are released before Gemma loads. No agreement, SFX or watermark filter is applied here.

In [4]:
def model_identity(path):
    return {str(p.relative_to(path)):[p.stat().st_size,p.stat().st_mtime_ns]
            for p in sorted(Path(path).rglob('*')) if p.is_file()
            and p.suffix in {'.json','.py','.bin','.safetensors','.pdparams','.pdiparams','.yml'}}

missing_pages=[p for p in PAGE_FILES if str(p) not in pages]
identity=None
if missing_pages:
    identity=dict(magi=str(MAGI_MODEL),magi_files=model_identity(MAGI_MODEL),
        paddle_det=str(PADDLE_DET_PATH),det_files=model_identity(PADDLE_DET_PATH),
        paddle_rec=str(PADDLE_REC_PATH),rec_files=model_identity(PADDLE_REC_PATH),algorithm=1)
    for path in list(missing_pages):
        cache_path=CACHE/(digest(dict(page=file_hash(path),models=identity))+'.json')
        if cache_path.is_file():
            try:
                pages[str(path)]=dict(records=read_payload(cache_path),source=str(cache_path))
                missing_pages.remove(path)
            except (OSError,ValueError,KeyError,AssertionError):pass

if missing_pages:
    from transformers import AutoConfig,AutoModel
    assert MAGI_MODEL.is_dir(),f'Missing Magi: {MAGI_MODEL}'
    config=AutoConfig.from_pretrained(str(MAGI_MODEL),trust_remote_code=True,local_files_only=True)
    config.detection_model_config.use_pretrained_backbone=False
    config.disable_ocr=False
    config.disable_detections=False
    model=AutoModel.from_pretrained(str(MAGI_MODEL),config=config,
        trust_remote_code=True,local_files_only=True).to(DEVICE).eval()
    raw_pages=[]
    try:
        for path in missing_pages:
            with Image.open(path) as im:array=np.asarray(im.convert('L').convert('RGB'))
            with torch.inference_mode():
                raw_pages.append(model.do_chapter_wide_prediction([array],{'images':[],'names':[]},do_ocr=True)[0])
    finally:
        del model
        gc.collect()
        if torch.cuda.is_available():torch.cuda.empty_cache()
    import paddle
    from paddleocr import PaddleOCR
    assert PADDLE_DET_PATH.is_dir() and PADDLE_REC_PATH.is_dir(),'Set local Paddle model directories.'
    engine=PaddleOCR(text_detection_model_name='PP-OCRv5_server_det',
        text_recognition_model_name='en_PP-OCRv5_mobile_rec',
        text_detection_model_dir=str(PADDLE_DET_PATH),text_recognition_model_dir=str(PADDLE_REC_PATH),
        device='gpu:0' if DEVICE=='cuda' else 'cpu',
        use_doc_orientation_classify=False,use_doc_unwarping=False,use_textline_orientation=False)
    try:
        for path,raw in zip(missing_pages,raw_pages):
            with Image.open(path) as im:im=im.convert('L').convert('RGB')
            records=[]
            for box,magi_text in zip(raw['texts'],raw['ocr']):
                a,b,c,d=box
                bounds=[max(0,math.floor(a)),max(0,math.floor(b)),min(im.width,math.ceil(c)),min(im.height,math.ceil(d))]
                lines=[];error=None
                try:
                    assert bounds[2]>bounds[0] and bounds[3]>bounds[1],'Invalid crop'
                    for prediction in engine.predict(input=np.asarray(im.crop(bounds))):
                        lines.extend(dict(text=str(t),confidence=float(s))
                            for t,s in zip(prediction['rec_texts'],prediction['rec_scores']))
                except Exception as exc:error=str(exc)
                records.append(dict(box=box,magi_text=magi_text,paddle_text=' '.join(v['text'] for v in lines),
                    paddle_lines=lines,selected_text=None,paddle_status='error' if error else 'text' if lines else 'empty',paddle_error=error))
            cache_path=CACHE/(digest(dict(page=file_hash(path),models=identity))+'.json')
            cache_path.write_text(json.dumps(dict(data=records,data_hash=digest(records)),indent=2,ensure_ascii=True),encoding='utf-8')
            pages[str(path)]=dict(records=records,source=str(cache_path))
            print('CREATED',path.name,cache_path)
    finally:
        del engine
        gc.collect()
        if DEVICE=='cuda':paddle.device.cuda.empty_cache()
print('Ready:',len(pages),'pages. Existing pipeline caches were not modified.')


Ready: 3 pages. Existing pipeline caches were not modified.


## 3. Define the classifier
Gemma receives only Paddle OCR and must return only Text or Artifact. Names, narration and punctuation count as Text. Sound effects, watermarks and nonsensical OCR count as Artifact. The punctuation rule runs in Python before generation. Low agreement is diagnostic only; it is never an automatic deletion in this test.
When Paddle OCR is empty, Magi punctuation-only text is retained directly as Text; other empty-Paddle boxes are marked Artifact/deleted before Gemma. `effective_text` and `text_source` record the chosen fallback.


In [5]:
SYSTEM_PROMPT=(
    'Classify the Paddle OCR text of one manga text box. Return exactly Text or Artifact. '
    'Text includes dialogue, narration, names, Japanese words/names, unfinished sentences, '
    'multiple sentences and multiple lines. Profanity and figurative language are allowed. '
    'Integers, integer-only text and numbers within meaningful dialogue are Text. '
    'Punctuation-only expressions are Text. Vocal reactions, screams, disgust, hesitation '
    'and stretched spoken words such as AAAAHH!!!, NOO!! and EWWW! are Text. '
    'Distinguish a human vocal reaction from a written environmental or action sound effect: '
    'Whoosh, BAM, CLANG and similar sound effects are Artifact. '
    'Artifact also includes nonsensical OCR noise, scan-group watermarks, website links '
    'and isolated code-like strings such as 2-B, 3-2 and V45S. '
    'Do not reject an unfamiliar name or a meaningful sentence simply because it contains '
    'a number or identifier. If uncertain, prefer Text. Do not correct, rewrite or explain the text.')

# One Paddle OCR input and one label per example.
EXAMPLES = [
    # Human vocal reactions and stretched dialogue are legitimate text.
    ('AAAAHH!!!', 'Text'),
    ('AAAH!', 'Text'),
    ('NOO!!', 'Text'),
    ('EWWW!', 'Text'),
    ('UGH... THAT SMELLS AWFUL!', 'Text'),
    ('HUH?!', 'Text'),
    ('OH...', 'Text'),
    ('UM... I THINK SO.', 'Text'),
    ('HAHAHA!', 'Text'),
    ('OW! THAT HURTS!', 'Text'),
    ('YAAAY!!', 'Text'),
    # Dialogue, narration, names, profanity and unfinished sentences.
    ('I SAID TO NOT DO THAT.', 'Text'),
    ('But our characters were unaware', 'Text'),
    ('NANIKAWA', 'Text'),
    ('SHIROGANE, WAIT!', 'Text'),
    ('もしもし', 'Text'),
    ("IT'S A GODDAMN MIRACLE, THE MONEY JUST VANISHED INTO THIN AIR.", 'Text'),
    ('IF SHE SUCCEEDS IN LURING OUT ONE PERSON IN THIS PLACE,', 'Text'),
    # Multiple sentences and OCR line breaks do not make a box an artifact.
    ('WAIT! WHERE DID EVERYONE GO? I WAS JUST HERE.', 'Text'),
    ('I TRUSTED YOU. NOW TELL ME WHAT HAPPENED.', 'Text'),
    ('WE HAVE TO LEAVE.\nTHE TRAIN IS ALREADY HERE!', 'Text'),
    ('Among the six who possessed the aptitude,\nSaiko-chan had the highest aptitude.', 'Text'),
    # Plain integers and numbers in meaningful lines are allowed.
    ('123', 'Text'),
    ('42', 'Text'),
    ('0', 'Text'),
    ('I HAVE 2 QUESTIONS.', 'Text'),
    ('WE HAVE 3 MINUTES LEFT!', 'Text'),
    ('HE PAID 500 YEN FOR IT.', 'Text'),
    ('I AM IN CLASS 2-B.', 'Text'),
    ('THE SCORE IS 3-2. WE CAN STILL WIN!', 'Text'),
    # Punctuation-only expressions are also protected by a Python rule.
    ('...', 'Text'),
    ('!!!!', 'Text'),
    ('...!!!', 'Text'),
    ('?!', 'Text'),
    # Written action/environment sounds, rather than human utterances.
    ('Whoosh!!', 'Artifact'),
    ('VOOSH', 'Artifact'),
    ('KIRK', 'Artifact'),
    ('bEE', 'Artifact'),
    ('WFAG', 'Artifact'),
    ('VOOOSH', 'Artifact'),
    ('O', 'Artifact'),
    ('K', 'Artifact'),
    ('AKK', 'Artifact'),
    ('HUPP', 'Artifact'),
    ('CRASHHH', 'Artifact'),
    ('SHHH', 'Artifact'),
    ('KRASHHH', 'Artifact'),
    ('BAM', 'Artifact'),
    ('CLANG!', 'Artifact'),
    ('THUD', 'Artifact'),
    ('CRASH!!', 'Artifact'),
    ('RUSTLE RUSTLE', 'Artifact'),
    ('CLICK', 'Artifact'),
    ('SPLASH!', 'Artifact'),
    ('SFX', 'Artifact'),
    ('SKX', 'Artifact'),
    # Watermarks, links and corrupted OCR.
    ('www.google.com', 'Artifact'),
    ('https://mangastream.com/read/123', 'Artifact'),
    ('MANGA STREAM DO NOT MIRROR', 'Artifact'),
    ('M4NGA STRE4M', 'Artifact'),
    ('DO NOT REDISTRIBUTE', 'Artifact'),
    ('oo', 'Artifact'),
    ('XZQWZXQ', 'Artifact'),
    # Standalone code-like strings requested as artifacts for this test.
    ('2-B', 'Artifact'),
    ('3-2', 'Artifact'),
    ('V45S', 'Artifact'),
]


def punctuation_only(text):
    text=str(text or '').strip()
    return bool(text) and all(c.isspace() or unicodedata.category(c).startswith('P') for c in text)


def direct_decision(paddle,magi=''):
    # Only punctuation may be recovered from Magi when Paddle finds no text.
    if not str(paddle or '').strip():
        if punctuation_only(magi):return 'Text','Magi punctuation fallback: Paddle OCR empty'
        return 'Artifact','Paddle OCR empty; no Magi punctuation fallback'
    if punctuation_only(paddle):return 'Text','punctuation-only override'
    return None,None


def paddle_prompt(paddle):
    return json.dumps({'Paddle OCR':str(paddle or '')},ensure_ascii=False)


def classify_paddle(paddle,model,tokenizer):
    messages=[dict(role='system',content=SYSTEM_PROMPT)]
    for example,label in EXAMPLES:
        messages.extend([dict(role='user',content=paddle_prompt(example)),dict(role='assistant',content=label)])
    messages.append(dict(role='user',content=paddle_prompt(paddle)))
    inputs=tokenizer.apply_chat_template(messages,add_generation_prompt=True,
        tokenize=True,return_dict=True,return_tensors='pt').to(model.device)
    with torch.inference_mode():
        output=model.generate(**inputs,do_sample=False,max_new_tokens=8)
    answer=tokenizer.decode(output[0,inputs['input_ids'].shape[1]:],skip_special_tokens=True).strip()
    # Unexpected model output stays visible as Review instead of causing deletion.
    label={'text':'Text','artifact':'Artifact'}.get(answer.casefold(),'Review')
    return dict(label=label,raw_answer=answer,reason='Gemma classification' if label!='Review' else 'unexpected Gemma answer')


## 4. Classify every box, reusing classification caches
Load local Gemma once in 4-bit NF4 only when uncached pairs remain. Cache keys include prompt/examples, model file identity and Paddle OCR text. Identical Paddle texts reuse the same decision. Red means Artifact, green Text, orange Review. Raw OCR is preserved.

In [6]:
MODEL_ID=model_identity(GEMMA_MODEL)
CLASSIFIER_SETTINGS=dict(model=str(GEMMA_MODEL),files=MODEL_ID,system=SYSTEM_PROMPT,
    examples=EXAMPLES,algorithm=3,input_source='paddle_only',empty_paddle='magi_punctuation_else_delete',max_new_tokens=8,quantization='nf4',
    transformers=__import__('transformers').__version__)
DECISION_CACHE=CACHE/'decisions'
DECISION_CACHE.mkdir(parents=True,exist_ok=True)
rows=[]
for path in PAGE_FILES:
    for index,record in enumerate(pages[str(path)]['records']):
        a=str(record.get('magi_text') or '');b=str(record.get('paddle_text') or '')
        na=''.join(re.findall('[a-z0-9]',a.casefold()))
        nb=''.join(re.findall('[a-z0-9]',b.casefold()))
        agreement=SequenceMatcher(None,na,nb).ratio() if na and nb else None
        rows.append(dict(page_file=str(path),box_id=f'PG{int(path.stem)}:T{index+1}',
            text_index=index,box=record['box'],magi_text=a,paddle_text=b,
            effective_text=b if b.strip() else a if punctuation_only(a) else '',
            text_source='paddle' if b.strip() else 'magi_punctuation' if punctuation_only(a) else 'none',
            paddle_lines=record.get('paddle_lines',[]),paddle_error=record.get('paddle_error'),
            agreement=agreement,low_agreement=agreement is not None and agreement<0.4,
            key=digest(dict(settings=CLASSIFIER_SETTINGS,paddle=b,magi_fallback=a if not b.strip() else None))))

pending={}
decisions={}
for row in rows:
    label,reason=direct_decision(row['paddle_text'],row['magi_text'])
    if label:
        decisions[row['key']]=dict(label=label,reason=reason,raw_answer='')
        continue
    cache_path=DECISION_CACHE/(row['key']+'.json')
    if cache_path.is_file() and not FORCE_GEMMA:
        try:
            saved=read_payload(cache_path)
            assert saved['label'] in {'Text','Artifact','Review'}
            decisions[row['key']]=saved
            continue
        except (OSError,ValueError,KeyError,AssertionError):pass
    pending[row['key']]=row

print('Boxes:',len(rows),'uncached unique Paddle texts:',len(pending))
if pending:
    assert GEMMA_MODEL.is_dir() and MODEL_ID,'Missing local Gemma model'
    assert torch.cuda.is_available(),'4-bit Gemma requires a CUDA kernel in this notebook'
    from transformers import AutoTokenizer,Gemma3ForConditionalGeneration,BitsAndBytesConfig
    dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    tokenizer=AutoTokenizer.from_pretrained(str(GEMMA_MODEL),local_files_only=True)
    model=Gemma3ForConditionalGeneration.from_pretrained(str(GEMMA_MODEL),local_files_only=True,
        device_map='auto',dtype=dtype,quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_compute_dtype=dtype)).eval()
    try:
        for number,(key,row) in enumerate(pending.items(),1):
            decision=classify_paddle(row['paddle_text'],model,tokenizer)
            decisions[key]=decision
            (DECISION_CACHE/(key+'.json')).write_text(json.dumps(
                dict(data=decision,data_hash=digest(decision)),ensure_ascii=False,indent=2),encoding='utf-8')
            print(number,'/',len(pending),row['box_id'],decision['label'])
    finally:
        del model,tokenizer
        gc.collect();torch.cuda.empty_cache()

for row in rows:
    row.update(decisions[row['key']])
    row['delete']=row['label']=='Artifact'
comparison=pd.DataFrame(rows,columns=['box_id','page_file','text_index','box','magi_text','paddle_text',
    'effective_text','text_source','paddle_lines','paddle_error','agreement','low_agreement','key','label','reason','raw_answer','delete'])
display(comparison[['box_id','magi_text','paddle_text','agreement','label','reason','raw_answer']])
print(comparison['label'].value_counts().to_dict())


Boxes: 24 uncached unique Paddle texts: 0


,box_id,magi_text,paddle_text,agreement,label,reason,raw_answer
0,PG1:T1,Wish,bE,0.000000,Artifact,Gemma classification,Artifact
1,PG1:T2,those aren't souvenirs.,THOSE AREN'T SOUVENIRS.,1.000000,Text,Gemma classification,Text
2,PG1:T3,They're for me to eat on the bullet train home.,THEYRE FOR ME TO EAT ON THE BULLET TRAIN HOME.,1.000000,Text,Gemma classification,Text
3,PG1:T4,behind you!,BEHIND YOU!,1.000000,Text,Gemma classification,Text
4,PG1:T5,"I've got a student watching, so...","IVE GOTA STUDENT WATCHING, sO...",1.000000,Text,Gemma classification,Text
5,PG1:T6,...I hope you don't mind if I show off a little.,...I HOPE YOU DON'T MIND IF I SHOW OFF A LITTLE.,1.000000,Text,Gemma classification,Text
6,PG1:T7,!!,!!,NaN,Text,punctuation-only override,
7,PG2:T1,The first time I was in the middle of the,,NaN,Artifact,Paddle OCR empty; no Magi punctuation fallback,
8,PG2:T2,I am,O,0.000000,Artifact,Gemma classification,Artifact
9,PG2:T3,.!!,,NaN,Text,Magi punctuation fallback: Paddle OCR empty,


{'Text': 17, 'Artifact': 7}


## 5. View pages and inspect individual crops
All boxes remain visible for comparison. Select **All pages** (the default) to view the entire selected sequence, or choose one page. The label filter applies to all displayed pages. Use the page dropdown and label filter, or call `show_page(0)` / `inspect_box("PG3:T4")` directly. Crop inspection shows both readings, agreement and the decision.

In [7]:
COLORS={'Text':'limegreen','Artifact':'red','Review':'darkorange'}


def show_page(page_index=0,label='All'):
    # All pages uses the same label filter for every page in the sequence.
    if page_index == -1:
        for index in range(len(PAGE_FILES)):
            show_page(index,label)
        return
    path=PAGE_FILES[page_index]
    with Image.open(path) as image:
        fig,ax=plt.subplots(figsize=(12,18))
        ax.imshow(image.convert('RGB'))
    for row in rows:
        if row['page_file']!=str(path) or label!='All' and row['label']!=label:continue
        x1,y1,x2,y2=row['box'];color=COLORS[row['label']]
        ax.add_patch(Rectangle((x1,y1),x2-x1,y2-y1,fill=False,edgecolor=color,linewidth=2))
        ax.text(x1,y1,row['box_id']+' '+row['label'],fontsize=8,
            bbox=dict(facecolor=color,alpha=.8),color='black')
    ax.set_title(path.name+' | green Text / red Artifact / orange Review')
    ax.axis('off');plt.show();plt.close(fig)


def inspect_box(box_id):
    row=next(r for r in rows if r['box_id']==box_id)
    with Image.open(row['page_file']) as image:
        x1,y1,x2,y2=row['box'];pad=12
        crop=image.crop((max(0,math.floor(x1)-pad),max(0,math.floor(y1)-pad),
            min(image.width,math.ceil(x2)+pad),min(image.height,math.ceil(y2)+pad)))
        fig,ax=plt.subplots(figsize=(10,4));ax.imshow(crop)
        ax.set_title(box_id+' | '+row['label'],color=COLORS[row['label']]);ax.axis('off');plt.show();plt.close(fig)
    display(pd.DataFrame([row])[['box_id','magi_text','paddle_text','effective_text','text_source','agreement','label','reason','raw_answer','paddle_error']])


# Clear the previous viewer when this cell is rerun, and detach old callbacks.
from IPython.display import clear_output
for name in ['page_widget','label_widget','box_widget']:
    old=globals().get(name)
    if old is not None:
        old.unobserve_all()
        old.close()
for name in ['page_output','box_output','viewer_ui']:
    old=globals().get(name)
    if old is not None:old.close()
clear_output(wait=True)

try:
    import ipywidgets as widgets
    if rows:
        page_widget=widgets.Dropdown(options=[('All pages',-1)]+[(p.name,i) for i,p in enumerate(PAGE_FILES)],
            value=-1,description='Page')
        label_widget=widgets.Dropdown(options=['All','Artifact','Text','Review'],description='Show')
        box_widget=widgets.Dropdown(options=[('Choose a crop',None)]+[(r['box_id'],r['box_id']) for r in rows],
            value=None,description='Inspect')
        page_output=widgets.Output()
        box_output=widgets.Output()

        def refresh_pages(change=None):
            # Replace this output area on each selection; do not append more pages.
            with page_output:
                clear_output(wait=True)
                show_page(page_widget.value,label_widget.value)

        def refresh_crop(change=None):
            with box_output:
                clear_output(wait=False)
                if box_widget.value is not None:inspect_box(box_widget.value)

        page_widget.observe(refresh_pages,names='value')
        label_widget.observe(refresh_pages,names='value')
        box_widget.observe(refresh_crop,names='value')
        viewer_ui=widgets.VBox([widgets.HBox([page_widget,label_widget]),page_output,box_widget,box_output])
        display(viewer_ui)
        refresh_pages()
    else:
        print('No text boxes to inspect.')
except ImportError:
    print('Use show_page(0) and inspect_box("PG3:T4") directly.')
    if PAGE_FILES:show_page(-1)


## 6. Save the review results
Full results retain every box, both OCR readings and raw Gemma answers. Separate Artifact and retained files are convenient for review; nothing is connected to production inference.

In [8]:
OUTPUT=CACHE/SPLIT/SEQUENCE_ID/('sequence' if PAGE_NUMBER is None else f'page_{PAGE_NUMBER:02d}')
OUTPUT.mkdir(parents=True,exist_ok=True)
comparison.to_csv(OUTPUT/'comparison.csv',index=False)
comparison.to_json(OUTPUT/'comparison.json',orient='records',force_ascii=False,indent=2)
comparison.loc[comparison['delete']].to_csv(OUTPUT/'artifacts.csv',index=False)
comparison.loc[~comparison['delete']].to_csv(OUTPUT/'retained.csv',index=False)
metadata=dict(settings=CLASSIFIER_SETTINGS,sequence_id=SEQUENCE_ID,split=SPLIT,
    pages=[dict(path=str(p),sha256=file_hash(p),ocr_source=pages[str(p)]['source']) for p in PAGE_FILES],
    counts=comparison['label'].value_counts().to_dict(),agreement_deletion=False,
    punctuation_override=True,kernel=sys.executable)
(OUTPUT/'metadata.json').write_text(json.dumps(metadata,ensure_ascii=False,indent=2),encoding='utf-8')
print('Saved:',OUTPUT)


Saved: C:\Users\ESHAAN\BYTE\Applied-Task\Magi\paddleOCR-Magi\cache_artifact_gemma_test\development\seq_5e4a9d63617d62bf\sequence
